In [17]:
import spacy
from SPARQLWrapper import SPARQLWrapper, JSON
from typing import List, Dict

class WikidataAnnotator:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_lg")
        self.sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
        self.sparql.setReturnFormat(JSON)

    def annotate_text(self, text: str) -> Dict:
        """Annotate text with Wikidata concepts"""
        doc = self.nlp(text)
        annotations = []

        # Simply get all named entities
        for ent in doc.ents:
            matches = self.search_wikidata(ent.text)
            if matches:
                annotations.append({
                    'text': ent.text,
                    'type': ent.label_,
                    'span': (ent.start_char, ent.end_char),
                    'wikidata_matches': matches
                })

        return {
            'text': text,
            'annotations': annotations
        }

    def search_wikidata(self, text: str) -> List[Dict]:
        """Basic Wikidata search"""
        query = f"""
        SELECT ?item ?itemLabel ?itemDescription WHERE {{
          ?item rdfs:label "{text}"@en.
          SERVICE wikibase:label {{ 
            bd:serviceParam wikibase:language "en".
          }}
          OPTIONAL {{ 
            ?item schema:description ?itemDescription.
            FILTER(LANG(?itemDescription) = "en")
          }}
        }}
        LIMIT 5
        """
        
        try:
            self.sparql.setQuery(query)
            results = self.sparql.query().convert()
            
            return [{
                'wikidata_id': r["item"]["value"].split("/")[-1],
                'label': r["itemLabel"]["value"],
                'description': r.get("itemDescription", {}).get("value", "")
            } for r in results["results"]["bindings"]]
        except:
            return []

# Example usage
if __name__ == "__main__":
    annotator = WikidataAnnotator()
    text = "Albert Einstein developed the theory of relativity at ETH Zurich in Switzerland."
    result = annotator.annotate_text(text)
    
    print("Original text:", result['text'])
    print("\nAnnotations:")
    for ann in result['annotations']:
        print(f"\nEntity: {ann['text']} ({ann['type']})")
        for match in ann['wikidata_matches']:
            print(f"- {match['label']} ({match['wikidata_id']})")
            print(f"  Description: {match['description']}")

Original text: Albert Einstein developed the theory of relativity at ETH Zurich in Switzerland.

Annotations:


In [35]:
import spacy
import requests

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

def search_wikidata(entity):
    """Search Wikidata for an entity and return the top match."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and len(data["search"]) > 0:
            result = data["search"][0]  # Take the top match
            return {
                "entity": entity,
                "wikidata_id": result["id"],
                "label": result.get("label", ""),
                "description": result.get("description", ""),
                "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
            }
    return None

def annotate_text(text):
    """Annotate text with Wikidata concepts."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract unique entities
    
    annotations = []
    for entity in entities:
        result = search_wikidata(entity)
        if result:
            annotations.append(result)

    return annotations

# Example Usage
if __name__ == "__main__":
    sentence = "Albert Einstein"
    results = annotate_text(sentence)
    
    # Display results
    for res in results:
        print(f"Entity: {res['entity']}")
        print(f"Wikidata ID: {res['wikidata_id']}")
        print(f"Label: {res['label']}")
        print(f"Description: {res['description']}")
        print(f"URL: {res['wikidata_url']}\n")


Entity: Albert Einstein
Wikidata ID: Q937
Label: Albert Einstein
Description: German-born theoretical physicist (1879–1955)
URL: https://www.wikidata.org/wiki/Q937



In [36]:
import spacy
import requests
import re

# Load spaCy's larger English model for better recognition
nlp = spacy.load("en_core_web_lg")

def search_wikidata(entity):
    """Search Wikidata for an entity and return top matches."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            results = []
            for result in data["search"]:
                results.append({
                    "entity": entity,
                    "wikidata_id": result["id"],
                    "label": result.get("label", entity),
                    "aliases": result.get("aliases", []),
                    "description": result.get("description", ""),
                    "wikidata_url": f"https://www.wikidata.org/wiki/{result['id']}"
                })
            return results  # Return all possible matches
    return None

def clean_entity(entity):
    """Clean entity by removing stop words and special characters."""
    entity = entity.lower().strip()
    entity = re.sub(r'[^\w\s]', '', entity)  # Remove punctuation
    return entity

def extract_entities(text):
    """Extract entities using both NER and keyword extraction."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract named entities
    
    # Extract additional keywords (noun chunks)
    for chunk in doc.noun_chunks:
        clean_chunk = clean_entity(chunk.text)
        if clean_chunk and len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)

def annotate_text(text):
    """Annotate text with Wikidata concepts."""
    entities = extract_entities(text)
    
    annotations = []
    for entity in entities:
        results = search_wikidata(entity)
        if results:
            annotations.extend(results)  # Append all possible matches

    return annotations

# Example Usage
if __name__ == "__main__":
    sentence = "Python is a computer programming language, and Tesla was founded by Elon Musk in the United States."
    results = annotate_text(sentence)
    
    # Display results
    for res in results:
        print(f"Entity: {res['entity']}")
        print(f"Wikidata ID: {res['wikidata_id']}")
        print(f"Label: {res['label']}")
        print(f"Aliases: {res.get('aliases', [])}")
        print(f"Description: {res['description']}")
        print(f"URL: {res['wikidata_url']}\n")


Entity: Python
Wikidata ID: Q28865
Label: Python
Aliases: []
Description: general-purpose programming language
URL: https://www.wikidata.org/wiki/Q28865

Entity: Python
Wikidata ID: Q76417859
Label: Python
Aliases: []
Description: family name
URL: https://www.wikidata.org/wiki/Q76417859

Entity: Python
Wikidata ID: Q184018
Label: pythons
Aliases: []
Description: family of snakes
URL: https://www.wikidata.org/wiki/Q184018

Entity: Python
Wikidata ID: Q271218
Label: Python
Aliases: []
Description: genus of reptiles
URL: https://www.wikidata.org/wiki/Q271218

Entity: Python
Wikidata ID: Q29642950
Label: Python package
Aliases: []
Description: software library for the Python programming language
URL: https://www.wikidata.org/wiki/Q29642950

Entity: Python
Wikidata ID: Q2120075
Label: Python
Aliases: []
Description: German ship
URL: https://www.wikidata.org/wiki/Q2120075

Entity: Python
Wikidata ID: Q599384
Label: CPython
Aliases: ['Python']
Description: Python reference implementation
URL:

In [44]:
import spacy
import requests
import re
from difflib import SequenceMatcher

# Load spaCy's larger English model for better entity recognition
nlp = spacy.load("en_core_web_lg")

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity):
    """Search Wikidata for an entity and return the best match based on relevance."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = None
            highest_score = 0.0
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]
                
                # Compute similarity score
                similarity = string_similarity(entity, label)
                
                # Combine with description match if available
                if entity.lower() in description.lower():
                    similarity += 0.1  # Small boost for descriptions matching
                
                if similarity > highest_score:
                    highest_score = similarity
                    best_match = {
                        "entity": entity,
                        "wikidata_id": wikidata_id,
                        "label": label,
                        "description": description,
                        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                        "relevance_score": round(highest_score, 3)  # Keep precision limited
                    }
            
            return best_match  # Return only the best match
    return None

def clean_entity(entity):
    """Clean entity by removing stop words and special characters."""
    entity = entity.lower().strip()
    entity = re.sub(r'[^\w\s]', '', entity)  # Remove punctuation
    return entity

def extract_entities(text):
    """Extract entities using both NER and keyword extraction."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract named entities
    
    # Extract additional keywords (noun chunks)
    for chunk in doc.noun_chunks:
        clean_chunk = clean_entity(chunk.text)
        if clean_chunk and len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)

def annotate_text(text):
    """Annotate text with Wikidata concepts using relevance filtering."""
    entities = extract_entities(text)
    
    annotations = []
    for entity in entities:
        result = search_wikidata(entity)
        if result and result["relevance_score"] > 0.7:  # Set a threshold for filtering
            annotations.append(result)

    return annotations

# Example Usage
if __name__ == "__main__":
    sentence = "Gotterdammerung"
    results = annotate_text(sentence)
    
    # Display results
    for res in results:
        print(f"Entity: {res['entity']}")
        print(f"Wikidata ID: {res['wikidata_id']}")
        print(f"Label: {res['label']}")
        print(f"Description: {res['description']}")
        print(f"Relevance Score: {res['relevance_score']}")
        print(f"URL: {res['wikidata_url']}\n")


Entity: Gotterdammerung
Wikidata ID: Q76551902
Label: Gotterdammerung
Description: print in the National Gallery of Art (NGA 149311)
Relevance Score: 1.0
URL: https://www.wikidata.org/wiki/Q76551902



In [45]:
import spacy
import requests
import re
import json
from difflib import SequenceMatcher

# Load spaCy's English model
nlp = spacy.load("en_core_web_lg")

def string_similarity(a, b):
    """Compute string similarity score using SequenceMatcher."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def search_wikidata(entity):
    """Search Wikidata for an entity and return the best match based on relevance."""
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "language": "en",
        "format": "json",
        "search": entity
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if "search" in data and data["search"]:
            best_match = None
            highest_score = 0.0
            
            for result in data["search"]:
                label = result.get("label", "")
                description = result.get("description", "")
                wikidata_id = result["id"]

                # Compute similarity score
                similarity = string_similarity(entity, label)

                # Final score
                final_score = similarity
                
                if final_score > highest_score:
                    highest_score = final_score
                    best_match = {
                        "entity": entity,
                        "wikidata_id": wikidata_id,
                        "label": label,
                        "description": description,
                        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_id}",
                        "relevance_score": round(final_score, 3)
                    }
            
            return best_match  # Return only the best match
    return None

def clean_entity(entity):
    """Clean entity by removing stop words and special characters."""
    entity = entity.lower().strip()
    entity = re.sub(r'[^\w\s]', '', entity)  # Remove punctuation
    return entity

def extract_entities(text):
    """Extract entities using both NER and keyword extraction."""
    doc = nlp(text)
    entities = set(ent.text for ent in doc.ents)  # Extract named entities
    
    # Extract additional keywords (noun chunks)
    for chunk in doc.noun_chunks:
        clean_chunk = clean_entity(chunk.text)
        if clean_chunk and len(clean_chunk) > 2:  # Avoid short words
            entities.add(chunk.text)

    return list(entities)

def annotate_text(text):
    """Annotate text with Wikidata concepts using general relevance filtering."""
    entities = extract_entities(text)
    
    annotations = []
    for entity in entities:
        result = search_wikidata(entity)
        if result and result["relevance_score"] > 0.7:  # Set a threshold for filtering
            annotations.append(result)

    output = {
        "original_sentence": text,
        "annotations": annotations
    }

    return json.dumps(output, indent=4)  # Convert to JSON format

# Example Usage
if __name__ == "__main__":
    sentence = "Python is a programming language, and Tesla was founded by Elon Musk in the United States."
    json_output = annotate_text(sentence)
    print(json_output)


{
    "original_sentence": "Python is a programming language, and Tesla was founded by Elon Musk in the United States.",
    "annotations": [
        {
            "entity": "a programming language",
            "wikidata_id": "Q105954505",
            "label": "A Programming Language",
            "description": "book by Iverson",
            "wikidata_url": "https://www.wikidata.org/wiki/Q105954505",
            "relevance_score": 1.0
        },
        {
            "entity": "Python",
            "wikidata_id": "Q28865",
            "label": "Python",
            "description": "general-purpose programming language",
            "wikidata_url": "https://www.wikidata.org/wiki/Q28865",
            "relevance_score": 1.0
        },
        {
            "entity": "Elon Musk",
            "wikidata_id": "Q317521",
            "label": "Elon Musk",
            "description": "South African-Canadian-American businessperson, engineer, inventor, philanthropist and political figure (born 19

# trying with SMILES integration

In [1]:
import re
import spacy
from typing import List, Dict, Set
import string

class ChEBI_NER:
    def __init__(self, obo_path: str):
        """
        Initializes the ChEBI Named Entity Recognizer by loading the full ontology.

        :param obo_path: Path to the ChEBI OBO ontology file.
        """
        # Common verbs to exclude even if they appear in ChEBI
        self.excluded_terms = {
            "is", "are", "was", "were", "be", "being", "been",
            "has", "have", "had", "having",
            "do", "does", "did", "doing",
            "can", "could", "may", "might", "must", "should", "would"
        }
        
        self.entity_map = self.load_chebi_ontology(obo_path)
        self.nlp = spacy.load("en_core_web_lg")
        
        # SMILES-specific patterns
        # This regex matches basic SMILES patterns - can be extended for more complex cases
        self.smiles_pattern = re.compile(r'\b[CNOS\[\]\(\)=#@\-\+1-9\\\/\.]+\b')

    def load_chebi_ontology(self, file_path: str) -> Dict:
        """
        Parses the ChEBI OBO file and extracts entity names with their ChEBI IDs.
        - Normalizes entity names for case-insensitive matching.
        - Handles alternative naming variations.
        - Excludes common verbs like "is" and "has"

        :param file_path: Path to the ChEBI OBO file.
        :return: Dictionary mapping entity names to ChEBI IDs.
        """
        entity_map = {}
        current_term = {}
        smiles_map = {}  # Map to store SMILES strings with their ChEBI IDs

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()

                if line == "[Term]":
                    if "id" in current_term and "name" in current_term:
                        term_id = current_term["id"]
                        term_name = current_term["name"].lower()

                        # Skip common verbs and auxiliary words
                        if term_name not in self.excluded_terms:
                            # Store the main name
                            entity_map[term_name] = term_id

                            # Store alternative variations (remove "atom", etc.)
                            simplified_name = re.sub(r'\s+atom$', '', term_name)
                            if simplified_name not in self.excluded_terms:
                                entity_map[simplified_name] = term_id
                        
                        # Add SMILES to map if it exists
                        if "smiles" in current_term:
                            smiles_map[current_term["smiles"]] = term_id
                    
                    current_term = {}

                elif line.startswith("id: CHEBI:"):
                    current_term["id"] = line.split(": ")[1].strip()

                elif line.startswith("name:"):
                    current_term["name"] = line.split(": ", 1)[1].strip()
                    
                elif line.startswith("synonym:"):
                    # Extract synonym from the line
                    synonym_match = re.search(r'"([^"]+)"', line)
                    if synonym_match:
                        synonym = synonym_match.group(1).lower()
                        if synonym not in self.excluded_terms:
                            entity_map[synonym] = current_term.get("id", "")
                
                # Extract SMILES notation if present
                elif line.startswith("property_value: chebi#smiles"):
                    smiles_match = re.search(r'"([^"]+)"', line)
                    if smiles_match:
                        current_term["smiles"] = smiles_match.group(1)

        print(f"✅ Loaded ChEBI ontology: {len(entity_map)} entities and {len(smiles_map)} SMILES patterns extracted.")
        self.smiles_map = smiles_map
        return entity_map

    def find_entities(self, text: str) -> List[Dict]:
        """
        Identifies both ChEBI entities and SMILES notation in a given text.
        
        :param text: Input sentence.
        :return: List of detected entities with their ChEBI IDs and/or SMILES notation.
        """
        detected_entities = []
        text_lower = text.lower()
        
        # Process text through spaCy for POS tagging
        doc = self.nlp(text)
        verb_tokens = [token for token in doc if token.pos_ == "VERB" or token.pos_ == "AUX"]
        verb_spans = [(token.idx, token.idx + len(token.text)) for token in verb_tokens]
        
        # Look for ChEBI entity matches, prioritizing longer terms
        for term, term_id in sorted(self.entity_map.items(), key=lambda x: len(x[0]), reverse=True):
            for match in re.finditer(rf'\b{re.escape(term)}\b', text_lower):
                start, end = match.span()
                
                # Skip if overlap with a verb
                is_verb = False
                for v_start, v_end in verb_spans:
                    if (start <= v_start and end > v_start) or (start < v_end and end >= v_end):
                        is_verb = True
                        break
                
                if is_verb:
                    continue
                
                # Get original text
                original_text = text[start:end]
                if original_text.islower():
                    original_text = original_text.capitalize()
                
                detected_entities.append({
                    "name": original_text,
                    "id": term_id,
                    "type": "ChEBI",
                    "span": (start, end)
                })
                break
        
        # Look for SMILES matches
        # First check if the SMILES is in our mapping
        for smiles, smiles_id in self.smiles_map.items():
            for match in re.finditer(re.escape(smiles), text):
                start, end = match.span()
                detected_entities.append({
                    "name": text[start:end],
                    "id": smiles_id,
                    "type": "SMILES_mapped",
                    "span": (start, end)
                })
        
        # Then look for generic SMILES patterns that might not be in our mapping
        for match in self.smiles_pattern.finditer(text):
            start, end = match.span()
            smiles_str = text[start:end]
            
            # Check if this looks like a valid SMILES string
            # This is a simple validation - you might want more sophisticated checks
            if self._is_valid_smiles(smiles_str):
                # Check if we haven't already found this as a mapped SMILES
                already_found = False
                for entity in detected_entities:
                    if entity.get("type") == "SMILES_mapped" and entity["span"] == (start, end):
                        already_found = True
                        break
                
                if not already_found:
                    detected_entities.append({
                        "name": smiles_str,
                        "id": None,  # No ChEBI ID for unmapped SMILES
                        "type": "SMILES_pattern",
                        "span": (start, end)
                    })
        
        # Filter overlapping entities, keeping longest ones
        detected_entities.sort(key=lambda x: (x["span"][0], -(x["span"][1] - x["span"][0])))
        
        non_overlapping = []
        last_end = -1
        
        for entity in detected_entities:
            start, end = entity["span"]
            if start >= last_end:
                non_overlapping.append(entity)
                last_end = end
        
        return non_overlapping
    
    def _is_valid_smiles(self, smiles: str) -> bool:
        """
        Basic validation for SMILES strings.
        
        :param smiles: SMILES string to validate
        :return: True if potentially valid, False otherwise
        """
        # Check for basic SMILES characteristics
        # This is a simplified validation - for production use a proper SMILES parser
        
        # Must contain at least one atom
        if not re.search(r'[CNOS]', smiles):
            return False
            
        # Basic check for balanced parentheses
        if smiles.count('(') != smiles.count(')'):
            return False
            
        # Basic check for balanced square brackets
        if smiles.count('[') != smiles.count(']'):
            return False
            
        # Should be longer than 2 characters for a meaningful molecule
        if len(smiles) < 3:
            return False
            
        return True

# Example Usage
if __name__ == "__main__":
    obo_path = "/home/matt/Proj/Hermeticav2/notebooks/prototyping/QAReasoning/chebi.obo"  # Replace with actual path
    ner = ChEBI_NER(obo_path)

    test_sentences = [
        "Water contains hydrogen and oxygen.",
        "Lactic acid is tautomer of pyruvic acid.",
        "Acetic acid is conjugate acid of acetate.",
        "Methanol has functional parent methane.",
        "Benzene has parent hydride cyclohexane.",
        "D-glucose is enantiomer of L-glucose.",
        "Caffeine has role psychoactive drug.",
        "The SMILES notation for caffeine is CN1C=NC2=C1C(=O)N(C)C(=O)N2C.",
        "Aspirin (C9H8O4) can be represented as CC(=O)OC1=CC=CC=C1C(=O)O in SMILES.",
        "CN1C=NC2=C1C(=O)N(C)C(=O)N2C is the chemical structure of caffeine."
    ]

    for sentence in test_sentences:
        print("\n==============================")
        print(f"🔬 Processing: \"{sentence}\"")
        entities = ner.find_entities(sentence)
        for entity in entities:
            entity_type = entity.get("type", "ChEBI")
            entity_id = entity["id"] if entity["id"] else "N/A"
            print(f"  - {entity['name']} ({entity_type}: {entity_id}) at position {entity['span']}")

✅ Loaded ChEBI ontology: 539620 entities and 0 SMILES patterns extracted.

🔬 Processing: "Water contains hydrogen and oxygen."
  - Water (ChEBI: CHEBI:15377) at position (0, 5)
  - Hydrogen (ChEBI: CHEBI:49637) at position (15, 23)
  - Oxygen (ChEBI: CHEBI:25805) at position (28, 34)

🔬 Processing: "Lactic acid is tautomer of pyruvic acid."
  - Lactic acid (ChEBI: CHEBI:28358) at position (0, 11)
  - Pyruvic acid (ChEBI: CHEBI:32816) at position (27, 39)

🔬 Processing: "Acetic acid is conjugate acid of acetate."
  - Acetic acid (ChEBI: CHEBI:15366) at position (0, 11)
  - Acetate (ChEBI: CHEBI:47622) at position (33, 40)

🔬 Processing: "Methanol has functional parent methane."
  - Methanol (ChEBI: CHEBI:17790) at position (0, 8)
  - Methane (ChEBI: CHEBI:16183) at position (31, 38)

🔬 Processing: "Benzene has parent hydride cyclohexane."
  - Benzene (ChEBI: CHEBI:16716) at position (0, 7)
  - Hydride (ChEBI: CHEBI:29239) at position (19, 26)
  - Cyclohexane (ChEBI: CHEBI:29005) at posi

In [3]:
import re
import spacy
from typing import List, Dict, Set
import string

class ChEBI_NER:
    def __init__(self, obo_path: str):
        """
        Initializes the ChEBI Named Entity Recognizer by loading the full ontology.

        :param obo_path: Path to the ChEBI OBO ontology file.
        """
        self.excluded_terms = {
            "is", "are", "was", "were", "be", "being", "been",
            "has", "have", "had", "having",
            "do", "does", "did", "doing",
            "can", "could", "may", "might", "must", "should", "would"
        }
        
        self.entity_map, self.smiles_map = self.load_chebi_ontology(obo_path)
        self.nlp = spacy.load("en_core_web_lg")
        
        # Regex for basic SMILES patterns
        self.smiles_pattern = re.compile(r'\b[CNOS\[\]\(\)=#@\-\+1-9\\\/\.]+\b')

    def load_chebi_ontology(self, file_path: str):
        """
        Parses the ChEBI OBO file and extracts entity names with their ChEBI IDs and SMILES.

        :param file_path: Path to the ChEBI OBO file.
        :return: Dictionary mapping entity names to ChEBI IDs, and a separate SMILES-to-ID map.
        """
        entity_map = {}
        smiles_map = {}
        chebi_to_smiles = {}  # To store bidirectional mapping
        current_term = {}

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()

                if line == "[Term]":
                    if "id" in current_term and "name" in current_term:
                        term_id = current_term["id"]
                        term_name = current_term["name"].lower()

                        if term_name not in self.excluded_terms:
                            entity_map[term_name] = term_id

                            # Store alternative naming variations
                            simplified_name = re.sub(r'\s+atom$', '', term_name)
                            if simplified_name not in self.excluded_terms:
                                entity_map[simplified_name] = term_id

                        # Store SMILES if available
                        if "smiles" in current_term:
                            smiles = current_term["smiles"]
                            smiles_map[smiles] = term_id
                            chebi_to_smiles[term_id] = smiles  # Bidirectional mapping

                    current_term = {}

                elif line.startswith("id: CHEBI:"):
                    current_term["id"] = line.split(": ")[1].strip()

                elif line.startswith("name:"):
                    current_term["name"] = line.split(": ", 1)[1].strip()
                    
                elif line.startswith("synonym:"):
                    synonym_match = re.search(r'"([^"]+)"', line)
                    if synonym_match:
                        synonym = synonym_match.group(1).lower()
                        if synonym not in self.excluded_terms:
                            entity_map[synonym] = current_term.get("id", "")

                elif line.startswith("property_value: chebi#smiles"):
                    smiles_match = re.search(r'"([^"]+)"', line)
                    if smiles_match:
                        current_term["smiles"] = smiles_match.group(1)

        print(f"✅ Loaded ChEBI ontology: {len(entity_map)} entities and {len(smiles_map)} SMILES patterns extracted.")
        self.chebi_to_smiles = chebi_to_smiles  # Store for reference
        return entity_map, smiles_map

    def find_entities(self, text: str) -> List[Dict]:
        """
        Identifies ChEBI entities and SMILES notation in a given text.
        
        :param text: Input sentence.
        :return: List of detected entities with their ChEBI IDs and/or SMILES notation.
        """
        detected_entities = []
        text_lower = text.lower()
        
        doc = self.nlp(text)
        verb_tokens = [token for token in doc if token.pos_ in {"VERB", "AUX"}]
        verb_spans = [(token.idx, token.idx + len(token.text)) for token in verb_tokens]

        # Look for ChEBI entity matches
        for term, term_id in sorted(self.entity_map.items(), key=lambda x: len(x[0]), reverse=True):
            for match in re.finditer(rf'\b{re.escape(term)}\b', text_lower):
                start, end = match.span()

                if any(start <= v_start < end or start < v_end <= end for v_start, v_end in verb_spans):
                    continue  # Skip if overlapping with a verb

                original_text = text[start:end]
                if original_text.islower():
                    original_text = original_text.capitalize()

                entity_info = {
                    "name": original_text,
                    "id": term_id,
                    "type": "ChEBI",
                    "span": (start, end)
                }

                # Attach SMILES if available
                if term_id in self.chebi_to_smiles:
                    entity_info["smiles"] = self.chebi_to_smiles[term_id]

                detected_entities.append(entity_info)
                break

        # Look for mapped SMILES
        for smiles, smiles_id in self.smiles_map.items():
            for match in re.finditer(re.escape(smiles), text):
                start, end = match.span()
                detected_entities.append({
                    "name": text[start:end],
                    "id": smiles_id,
                    "type": "SMILES_mapped",
                    "span": (start, end),
                    "chebi_name": self.get_chebi_name(smiles_id)  # Attach ChEBI entity name if available
                })

        # Look for generic SMILES patterns
        for match in self.smiles_pattern.finditer(text):
            start, end = match.span()
            smiles_str = text[start:end]

            if self._is_valid_smiles(smiles_str) and smiles_str not in self.smiles_map:
                detected_entities.append({
                    "name": smiles_str,
                    "id": None,  
                    "type": "SMILES_pattern",
                    "span": (start, end)
                })

        # Ensure no overlaps
        detected_entities.sort(key=lambda x: (x["span"][0], -(x["span"][1] - x["span"][0])))

        non_overlapping = []
        last_end = -1
        for entity in detected_entities:
            if entity["span"][0] >= last_end:
                non_overlapping.append(entity)
                last_end = entity["span"][1]

        return non_overlapping
    
    def get_chebi_name(self, chebi_id: str) -> str:
        """Retrieve ChEBI entity name from ID, if available."""
        for name, id_value in self.entity_map.items():
            if id_value == chebi_id:
                return name
        return "Unknown"

    def _is_valid_smiles(self, smiles: str) -> bool:
        """Basic validation for SMILES strings."""
        if not re.search(r'[CNOS]', smiles):
            return False
        if smiles.count('(') != smiles.count(')'):
            return False
        if smiles.count('[') != smiles.count(']'):
            return False
        if len(smiles) < 3:
            return False
        return True

# Example Usage
if __name__ == "__main__":
    obo_path = "/home/matt/Proj/Hermeticav2/notebooks/prototyping/QAReasoning/chebi.obo"  
    ner = ChEBI_NER(obo_path)

    test_sentences = [
        "Caffeine has the SMILES CN1C=NC2=C1C(=O)N(C)C(=O)N2C.",
        "Acetic acid (ChEBI:15366) has the SMILES CC(=O)O.",
        "Methanol (CH3OH) is commonly used as a solvent."
    ]

    for sentence in test_sentences:
        print("\n==============================")
        print(f"🔬 Processing: \"{sentence}\"")
        entities = ner.find_entities(sentence)
        for entity in entities:
            print(f"  - {entity}")


✅ Loaded ChEBI ontology: 539620 entities and 0 SMILES patterns extracted.

🔬 Processing: "Caffeine has the SMILES CN1C=NC2=C1C(=O)N(C)C(=O)N2C."
  - {'name': 'Caffeine', 'id': 'CHEBI:27732', 'type': 'ChEBI', 'span': (0, 8)}
  - {'name': 'CN1C=NC2=C1C(=O)N(C)C(=O)N2C', 'id': None, 'type': 'SMILES_pattern', 'span': (24, 52)}

🔬 Processing: "Acetic acid (ChEBI:15366) has the SMILES CC(=O)O."
  - {'name': 'Acetic acid', 'id': 'CHEBI:15366', 'type': 'ChEBI', 'span': (0, 11)}
  - {'name': 'CC(=O)O', 'id': None, 'type': 'SMILES_pattern', 'span': (41, 48)}

🔬 Processing: "Methanol (CH3OH) is commonly used as a solvent."
  - {'name': 'Methanol', 'id': 'CHEBI:17790', 'type': 'ChEBI', 'span': (0, 8)}
  - {'name': 'CH3OH', 'id': 'CHEBI:17790', 'type': 'ChEBI', 'span': (10, 15)}
  - {'name': 'As', 'id': 'CHEBI:27563', 'type': 'ChEBI', 'span': (34, 36)}
  - {'name': 'A', 'id': 'CHEBI:13193', 'type': 'ChEBI', 'span': (37, 38)}
  - {'name': 'Solvent', 'id': 'CHEBI:46787', 'type': 'ChEBI', 'span': (39,

In [5]:
import re
import spacy
from typing import List, Dict, Set
import string

class ChEBI_NER:
    def __init__(self, obo_path: str):
        """
        Initializes the ChEBI Named Entity Recognizer by loading the full ontology.

        :param obo_path: Path to the ChEBI OBO ontology file.
        """
        self.excluded_terms = {
            "is", "are", "was", "were", "be", "being", "been",
            "has", "have", "had", "having",
            "do", "does", "did", "doing",
            "can", "could", "may", "might", "must", "should", "would"
        }
        
        self.entity_map, self.smiles_map, self.chebi_to_smiles = self.load_chebi_ontology(obo_path)
        self.nlp = spacy.load("en_core_web_lg")
        
        # Regex for basic SMILES patterns
        self.smiles_pattern = re.compile(r'\b[CNOS\[\]\(\)=#@\-\+1-9\\\/\.]+\b')

    def load_chebi_ontology(self, file_path: str):
        """
        Parses the ChEBI OBO file and extracts entity names with their ChEBI IDs and SMILES.

        :param file_path: Path to the ChEBI OBO file.
        :return: Dictionary mapping entity names to ChEBI IDs, SMILES-to-ID map, and ID-to-SMILES map.
        """
        entity_map = {}
        smiles_map = {}
        chebi_to_smiles = {}  
        current_term = {}

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()

                if line == "[Term]":
                    if "id" in current_term and "name" in current_term:
                        term_id = current_term["id"]
                        term_name = current_term["name"].lower()

                        if term_name not in self.excluded_terms:
                            entity_map[term_name] = term_id

                        if "smiles" in current_term:
                            smiles = current_term["smiles"]
                            smiles_map[smiles] = term_id
                            chebi_to_smiles[term_id] = smiles  # Store SMILES for ChEBI ID

                    current_term = {}

                elif line.startswith("id: CHEBI:"):
                    current_term["id"] = line.split(": ")[1].strip()

                elif line.startswith("name:"):
                    current_term["name"] = line.split(": ", 1)[1].strip()
                    
                elif line.startswith("synonym:"):
                    synonym_match = re.search(r'"([^"]+)"', line)
                    if synonym_match:
                        synonym = synonym_match.group(1).lower()
                        if synonym not in self.excluded_terms:
                            entity_map[synonym] = current_term.get("id", "")

                elif line.startswith("property_value: chebi#smiles"):
                    smiles_match = re.search(r'"([^"]+)"', line)
                    if smiles_match:
                        current_term["smiles"] = smiles_match.group(1)

        print(f"✅ Loaded ChEBI ontology: {len(entity_map)} entities and {len(smiles_map)} SMILES patterns extracted.")
        return entity_map, smiles_map, chebi_to_smiles

    def find_entities(self, text: str) -> List[Dict]:
        """
        Identifies ChEBI entities and SMILES notation in a given text.
        
        :param text: Input sentence.
        :return: List of detected entities with their ChEBI IDs and/or SMILES notation.
        """
        detected_entities = []
        text_lower = text.lower()
        
        doc = self.nlp(text)
        verb_tokens = [token for token in doc if token.pos_ in {"VERB", "AUX"}]
        verb_spans = [(token.idx, token.idx + len(token.text)) for token in verb_tokens]

        # Look for ChEBI entity matches
        for term, term_id in sorted(self.entity_map.items(), key=lambda x: len(x[0]), reverse=True):
            for match in re.finditer(rf'\b{re.escape(term)}\b', text_lower):
                start, end = match.span()

                if any(start <= v_start < end or start < v_end <= end for v_start, v_end in verb_spans):
                    continue  # Skip if overlapping with a verb

                original_text = text[start:end]
                if original_text.islower():
                    original_text = original_text.capitalize()

                entity_info = {
                    "name": original_text,
                    "id": term_id,
                    "type": "ChEBI",
                    "span": (start, end),
                    "smiles": self.get_smiles_by_chebi_id(term_id)  # Attach SMILES if available
                }

                detected_entities.append(entity_info)
                break

        # Look for mapped SMILES
        for smiles, smiles_id in self.smiles_map.items():
            for match in re.finditer(re.escape(smiles), text):
                start, end = match.span()
                detected_entities.append({
                    "name": text[start:end],
                    "id": smiles_id,
                    "type": "SMILES_mapped",
                    "span": (start, end),
                    "chebi_name": self.get_chebi_name(smiles_id)  # Attach ChEBI entity name if available
                })

        # Look for generic SMILES patterns
        for match in self.smiles_pattern.finditer(text):
            start, end = match.span()
            smiles_str = text[start:end]

            if self._is_valid_smiles(smiles_str) and smiles_str not in self.smiles_map:
                detected_entities.append({
                    "name": smiles_str,
                    "id": None,  
                    "type": "SMILES_pattern",
                    "span": (start, end)
                })

        # Ensure no overlaps
        detected_entities.sort(key=lambda x: (x["span"][0], -(x["span"][1] - x["span"][0])))

        non_overlapping = []
        last_end = -1
        for entity in detected_entities:
            if entity["span"][0] >= last_end:
                non_overlapping.append(entity)
                last_end = entity["span"][1]

        return non_overlapping
    
    def get_smiles_by_chebi_id(self, chebi_id: str) -> str:
        """Retrieve SMILES notation for a given ChEBI ID."""
        return self.chebi_to_smiles.get(chebi_id, "Unknown")

    def get_chebi_name(self, chebi_id: str) -> str:
        """Retrieve ChEBI entity name from ID, if available."""
        for name, id_value in self.entity_map.items():
            if id_value == chebi_id:
                return name
        return "Unknown"

    def _is_valid_smiles(self, smiles: str) -> bool:
        """Basic validation for SMILES strings."""
        if not re.search(r'[CNOS]', smiles):
            return False
        if smiles.count('(') != smiles.count(')'):
            return False
        if smiles.count('[') != smiles.count(']'):
            return False
        if len(smiles) < 3:
            return False
        return True

# Example Usage
if __name__ == "__main__":
    obo_path = "/home/matt/Proj/Hermeticav2/notebooks/prototyping/QAReasoning/chebi.obo"   
    ner = ChEBI_NER(obo_path)

    test_sentences = [
        "Caffeine has the SMILES CN1C=NC2=C1C(=O)N(C)C(=O)N2C.",
        "Acetic acid (ChEBI:15366) has the SMILES CC(=O)O.",
        "Methanol (CH3OH) is commonly used as a solvent.",
        "CHEBI:15377 corresponds to water.",
        "CHEBI:17347 is the ID for ethanol."
    ]

    for sentence in test_sentences:
        print("\n==============================")
        print(f"🔬 Processing: \"{sentence}\"")
        entities = ner.find_entities(sentence)
        for entity in entities:
            print(f"  - {entity}")


✅ Loaded ChEBI ontology: 539613 entities and 0 SMILES patterns extracted.

🔬 Processing: "Caffeine has the SMILES CN1C=NC2=C1C(=O)N(C)C(=O)N2C."
  - {'name': 'Caffeine', 'id': 'CHEBI:27732', 'type': 'ChEBI', 'span': (0, 8), 'smiles': 'Unknown'}
  - {'name': 'CN1C=NC2=C1C(=O)N(C)C(=O)N2C', 'id': None, 'type': 'SMILES_pattern', 'span': (24, 52)}

🔬 Processing: "Acetic acid (ChEBI:15366) has the SMILES CC(=O)O."
  - {'name': 'Acetic acid', 'id': 'CHEBI:15366', 'type': 'ChEBI', 'span': (0, 11), 'smiles': 'Unknown'}
  - {'name': 'CC(=O)O', 'id': None, 'type': 'SMILES_pattern', 'span': (41, 48)}

🔬 Processing: "Methanol (CH3OH) is commonly used as a solvent."
  - {'name': 'Methanol', 'id': 'CHEBI:17790', 'type': 'ChEBI', 'span': (0, 8), 'smiles': 'Unknown'}
  - {'name': 'CH3OH', 'id': 'CHEBI:17790', 'type': 'ChEBI', 'span': (10, 15), 'smiles': 'Unknown'}
  - {'name': 'As', 'id': 'CHEBI:27563', 'type': 'ChEBI', 'span': (34, 36), 'smiles': 'Unknown'}
  - {'name': 'A', 'id': 'CHEBI:13193', 'typ